# Upload the Kimi-style answer column to Hugging Face

CPU Colab is sufficient. Run the cells in order with a write-capable `HF_TOKEN`
in Colab Secrets. No DeepSeek calls or training are performed.

Source: Drive regeneration run **63d2e34487188421**, also used by the answer-only
EXAONE SFT notebook. Copy `deepseek-v4-pro_answer` verbatim into the new column
**`kimi-style-reasoning-answer`** in `Seungjun/dp_removed_s1K-1.1` (`default/train`).
This is the entire answer text, including its reasoning sections and final answer;
the separate API reasoning channel is not concatenated.

Keep all rows and existing HF columns, including the older Pro outputs.
Failed generation rows retain null in the new column. No length or correctness
filtering is applied. Existing grades still refer to their original answers.
The upload cell publishes to the existing repository with its existing visibility.
This version also repairs the stale feature metadata from an earlier upload.
If that upload failed verification with a column-name CastError, rerun this
notebook from the top in a fresh CPU runtime; no regeneration is needed.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "PyYAML==6.0.3"])
from google.colab import drive, userdata
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Enable a write-capable HF_TOKEN in Colab Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-kimi-answer-upload")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVzwUxbvewQAAEkMAAAfAAAAcGlwZWxpbmUvcHVibGlzaF9raW1pX2Fuc3dlci5weY1WTY/bOAy951dodZIB10UX2EuAHHrooYdtC3Sxe0iDQLDpRKgtuZI86exg/vtSpOw4nUyyuSTRB/n4+EhRSvm+aYQWHg5gwesI+M+GE3hRu27srTiZeHRjxBNDp2tjDwJ+mhDTj/BONDrqSkq5WrXe9fQ3QAzC9IPzUfytuxHyXu363tnKuGmzMQcIcbVaNdCKHvwB9uxaBTf6GkpRj96DjSW6TDegKUWIOo4BAm4625pDsV4J/PCNvXenIDaiQ4DZSMHbA9S4zle2krfkjvZqN9qYLoFVCzN80bR5/7cNGdlKhAI1QqFDciecz4FcXD6fR58RY9jzIbljvOnjtQnAFH3w3nklvxhrMQFsCM22LfggiL14BDG5nmiWM0ZtH1XCP/tm0C2CI0qMFeoGmUVxC9XX69ko0WuTTSQv7DOIfgxR9DrWx4yPhVROujqnwcJpz5tyV86rfGzeOKeB5IiRMD0VL+yt7iHcgv8JTtNlgmbdJGZASJg/czBWdxPr2S9Df0h2kqK2jCMRakqhXIfsdeYBWcEgJiKLhA7s2FMhqX/NsNTEHTmH6E0dN3/5EZbpyMn9Do8EHB0kt0l1iyXEkFbSzhaXd0kBBD2t4ubl4mw7f1o2VuYraA/Dq0yEPqgllKv0tvLzRGBroGuCYM3hVwM+a1joSGtP5jkTm0Pj8KsDRJWLcm9sAz8lidjc9f31rD7dmYPtkd65bl5zmnV45u6umz9NCKnhXXbGq/aZww2xzsd3rwZMvzHUzUZgm+iHDiLISzB4RRFQ9I/nbQ2KPJBeCkov7tJSlRQ0qOLX9NIHDy5dt8aacNx70MFZJhvhuEH+n8sewuBsgH3vGuhk7jdcvLy0+0U1r9D60SJw04gp9vnpucosdEhGlmigsD85C3dz99FO5ieb3AaOGstICzt2XXZ7qZJUTBQvZ7tYOqXOR9XGm+faugvnHzd2jXAP4E8eSyx1oPk9vRU9d6JKDwPYhjUwd/8EKzWJF+RQac9VzSZutvoP1JhSNqNGwaGhF8kJGRQ92E3q5tzVKg89hpX7djjTtmzd09ll78bEIoq8c2maf1S6abJZNb0lHEspWkBdethQEKmkPBIpC4boAfdstlKKJ0lP9pofqlLIKaC1CGOvHl4yt2CtvFoZMslnf8XOdRti+eStp4fxuuX8xk9zw3qaMmi24ZCK4jlPTxFTafeD9j9GvNKaDoLCN87xz5xyHNPe1zUMUTjbPdJE8d26k51TLNCUHrv4lsyJcNQeZa4fcfyjGY/kxrObx6ku/R10PKb3MdB7pgaKeUgxn90nCQzYnbSPIU2TimJ7i40jldFQoaDzepUjmBKIfwMJYeuhapFqGiuUZwOE8o361jz98Vy8ce3089tsphRDcUZEWHfLomH0uYnqrlPs8F6J8BBGJGbGOAHiC/tl5oIYLT2BHOzkV6VKNRZzuP19R70zDW6EhLH2jJWAXEgD97Z88d3uxVFqQaQNr+0B1NlocXu6W/THNMqOQ2dqnF6uRZS1IC+qi5ys/gNQSwMEFAAAAAgAAAAhXKvUjLlwAQAA/wIAACAAAABjb25maWdzL3B1Ymxpc2hfa2ltaV9hbnN3ZXIueWFtbLWSS27bMBCG9z6FLiBTfIh67AqkKIomKx9AoMmhyoQiCT7s+PalYyOLBkiRRVckZ/6Zb/4Bky9RwrxrmgjBz80Bilufi0MqLBE2fwK1JPyrxXv8pjmZZLybG67wMBw17wSeeiWpJEdGFXAxgtaYsK4XWE6M1CLpnTbr3CjQothcIylYk+cmR2FcfcJrAJkrKPpzmptp4reqDC4vyqyQqpgPPSO6B02JnmDQdOg7QY9TRUHPOj6oAeTQdQxzImBUVEjSkYlPZBKa036XRVwhf93pdhvyHy42r8BesxASwEt7Ym2IfidcOkNcpLdlcx/Syy29c3B+l7yYzbQpXyy0EUTyzri1veu0sZCuDpTIIlUzDbqvCaloToCeLg9v5+OP9tvPp+/tAbcPFXmoSBSLS+ivARCnigBlbBzwODKCkS7Wonv7fRAxG2H3z3UMe3WcRS7pf1Fv3d9hESSYcP0A9ZZ9vHyJ+1svJVgvVEKfbfQPUEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABIAAAAAAAAAAAAAAIABAAAAAGNvbW1vbi9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVx0Lccl3AIAAIYIAAAMAAAAAAAAAAAAAACAATIAAABjb21tb24vaW8ucHlQSwECFAMUAAAACAAAACFcAAAAAAIAAAAAAAAAFAAAAAAAAAAAAAAAgAE4AwAAcGlwZWxpbmUvX19pbml0X18ucHlQSwECFAMUAAAACAAAACFc8FMW73sEAABJDAAAHwAAAAAAAAAAAAAAgAFsAwAAcGlwZWxpbmUvcHVibGlzaF9raW1pX2Fuc3dlci5weVBLAQIUAxQAAAAIAAAAIVyr1Iy5cAEAAP8CAAAgAAAAAAAAAAAAAACAASQIAABjb25maWdzL3B1Ymxpc2hfa2ltaV9hbnN3ZXIueWFtbFBLBQYAAAAABQAFAFcBAADSCQAAAAA='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

In [ ]:
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/publish_kimi_answer.yaml").read_text())
# Plain settings: edit only if you moved the source files.
DATASET_PATH = CONFIG["files"]["dataset"]
STATUS_PATH = CONFIG["files"]["status"]
print("Source:", DATASET_PATH)
print("Target:", CONFIG["target"]["repo"])
print("Added column:", CONFIG["new_column"])

## Validate and preview

Compare every original field with the pinned source and current Hub data before
merging. An identical rerun is allowed; conflicting existing values are rejected.
All other current Hub columns remain intact. The preview shows up to three answers.

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path
from datasets import DatasetDict, get_dataset_split_names, load_dataset
from huggingface_hub import HfApi, hf_hub_download
from common.io import digest, read_jsonl, write_json
from pipeline.publish_kimi_answer import merge_answer, train_parquet_files

# Snapshot bytes once so the validated inputs cannot change mid-upload.
snapshot = Path("/content/kimi-answer-upload-inputs")
snapshot.mkdir(parents=True, exist_ok=True)
hashes = {}
for filename, original in [("dataset.jsonl", DATASET_PATH), ("status.jsonl", STATUS_PATH)]:
    content = Path(original).read_bytes()
    (snapshot / filename).write_bytes(content)
    hashes[filename] = hashlib.sha256(content).hexdigest()
exported = read_jsonl(snapshot / "dataset.jsonl")
statuses = read_jsonl(snapshot / "status.jsonl")
api = HfApi(token=HF_TOKEN)
spec, target = CONFIG["source"], CONFIG["target"]
before = api.dataset_info(target["repo"], revision=target["revision"]).sha
source = load_dataset(spec["repo"], name=spec["config"], split=spec["split"],
                      revision=spec["revision"], token=HF_TOKEN)
# This uploader targets the existing default/train-only repository.
# Read actual Parquet schemas: older uploads may have left stale README features.
if target["config"] != "default" or target["split"] != "train":
    raise ValueError("This uploader expects default/train")
splits = get_dataset_split_names(target["repo"], config_name=target["config"],
                                 revision=before, token=HF_TOKEN)
if splits != ["train"]:
    raise ValueError(f"Unexpected splits; refusing to replace them: {splits}")
paths = train_parquet_files(api.list_repo_files(
    target["repo"], repo_type="dataset", revision=before))
local_parquets = [hf_hub_download(target["repo"], filename=path, repo_type="dataset",
                                 revision=before, token=HF_TOKEN) for path in paths]
current = load_dataset("parquet", data_files={"train": local_parquets}, split="train")
merged, summary = merge_answer(source, current, exported, statuses, CONFIG)
receipt_dir = Path(CONFIG["files"]["receipt_directory"])
receipt_dir.mkdir(parents=True, exist_ok=True)
receipt_path = receipt_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
receipt = {"config": CONFIG, "dataset_path": DATASET_PATH, "status_path": STATUS_PATH,
           "input_sha256": hashes, "previous_revision": before, "summary": summary,
           "verified": False}
write_json(receipt_path, receipt)
print(json.dumps(summary, indent=2))
print("Preserved columns:", current.column_names)
print("Drive receipt:", receipt_path)
shown = 0
for i, row in enumerate(merged):
    if row[CONFIG["new_column"]] is not None:
        print("\nROW", i, "\nQUESTION:", row["question"])
        print("ANSWER:\n", row[CONFIG["new_column"]])
        shown += 1
        if shown == 3:
            break

## Upload

This cell writes the prepared column to Hugging Face. Use one uploader at a time.
If the repository changes after validation, rerun the validation cell first.

In [ ]:
if api.dataset_info(target["repo"], revision=target["revision"]).sha != before:
    raise RuntimeError("HF changed since validation; rerun validation before upload")
# DatasetDict refreshes the feature schema as well as the Parquet files.
commit = DatasetDict({"train": merged}).push_to_hub(
    target["repo"], config_name=target["config"],
    revision=target["revision"], token=HF_TOKEN,
    commit_message="Add kimi-style-reasoning-answer from regeneration 63d2e34487188421")
receipt["dataset_revision"] = commit.oid
write_json(receipt_path, receipt)
print("Uploaded revision:", commit.oid)
print("Receipt:", receipt_path)

## Verify the uploaded revision

In [ ]:
receipt = json.loads(receipt_path.read_text())
uploaded = load_dataset(target["repo"], name=target["config"], split=target["split"],
                        revision=receipt["dataset_revision"], token=HF_TOKEN)
if digest(list(uploaded)) != summary["dataset_digest"]:
    raise ValueError("Uploaded dataset differs from the prepared data")
receipt["verified"] = True
write_json(receipt_path, receipt)
print("Verified all rows and columns:", len(uploaded))
print("Pinned HF revision:", receipt["dataset_revision"])
print("Saved receipt:", receipt_path)
print(f"https://huggingface.co/datasets/{target['repo']}")